In [1]:
%reload_ext autoreload
%autoreload 2

import os
import sys
import dill as pickle
import argparse

import numpy as np
import healpy as hp

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

wdir = "/n/home07/yitians/fermi/fermi-prob-prog/production"
sys.path.append(f"{wdir}/..")
from models.np_model import NPModel
from models.np_model_1b import NPModel1B

/n/home07/yitians/.conda/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Args:
    pass
args = Args()
args.data = "base23fix"
args.i = 0
args.model = "base23fix_7exp"
args.n_exp = 7
args.truth_name = 'base230927'

In [3]:
mask_roi = np.load(f"{wdir}/mask_roi.npy")
mask_norm = jnp.load(f"{wdir}/mask_norm.npy")

# Ensure mask_roi's length is divisible by args.n_exp
n_pix_remainder = int(np.sum(~mask_roi)) % args.n_exp
print(f'Pixel number: {int(np.sum(~mask_roi))}', end=' ')
if n_pix_remainder != 0:
    unmasked_indices = np.where(mask_roi == 0)[0]
    mask_roi[unmasked_indices[-n_pix_remainder:]] = 1
print(f'-> {int(np.sum(~mask_roi))} = {args.n_exp} * {int(np.sum(~mask_roi) / args.n_exp)}')

data = np.load(f"../outputs/sims/{args.data}.npy")[args.i]
if len(data) < hp.nside2npix(128):
    data_full = np.zeros(hp.nside2npix(128))
    data_full[~mask_norm] = data
    data_in = jnp.array(data_full, dtype=jnp.int32)
else:
    data_in = jnp.array(data, dtype=jnp.int32)

psf_tag = 'delta' if 'deltapsf' in args.model else 'king'
print('PSF:', psf_tag)
if '1b' in args.model:
    Model = NPModel1B
    print('Using NPModel1B model')
else:
    Model = NPModel
    print('Using NPModel model')
m = Model(data=data_in, psf_tag=psf_tag, n_exp=args.n_exp, custom_mask_roi=mask_roi)

Pixel number: 6839 -> 6839 = 7 * 977
PSF: king
Using NPModel model
Number of exposure regions: 7
Number of pixels in ROI: 6839
Using psf: king
Loading the psf correction from: /n/home07/yitians/fermi/fermi-prob-prog/production/psf_dir/Fermi_PSF_2GeV2_nside128.npy
Max photon count is 109


In [8]:
map_est = m.get_MAP_estimates(n_steps=6000, data=data_in, lr=0.1)

100%|██████████| 6000/6000 [02:31<00:00, 39.65it/s, init loss: 22610.1496, avg. loss [5701-6000]: 16060.2420]


In [10]:
pickle.dump(m.MAP_estimates, open("tmp.p", "wb"))